# Load networks from an OSM PBF file

Author: [Geoff Boeing](https://geoffboeing.com/)

Load a bicycle network from a local PBF file using a built-in network type or custom osmium filters.

  - [Documentation](https://osmnx.readthedocs.io/)
  - [Journal article and citation info](https://doi.org/10.1111/gean.70009)
  - [Code repository](https://github.com/gboeing/osmnx)
  - [Examples gallery](https://github.com/gboeing/osmnx-examples)

In [ ]:
# !uv pip install --system --quiet osmnx[all]
from pathlib import Path
from urllib.request import urlretrieve

import osmium
import osmnx as ox

ox.settings.log_console = True
ox.__version__

## Download a PBF file

Download the District of Columbia extract from [Geofabrik](https://download.geofabrik.de/north-america/us/district-of-columbia.html), with data from [OpenStreetMap contributors](https://www.openstreetmap.org/copyright). The remaining examples use this local file.

In [ ]:
# download the source file and save it in the data folder
source = Path("./data/dc.osm.pbf")
source.parent.mkdir(parents=True, exist_ok=True)
url = "https://download.geofabrik.de/north-america/us/district-of-columbia-latest.osm.pbf"
_ = urlretrieve(url, source)

## 1. Use a built-in network type

Use `network_type="bike"` to select streets and paths for cycling. By default, OSMnx simplifies the topology and keeps the largest weakly connected component.

In [ ]:
G = ox.graph.graph_from_pbf(source, network_type="bike")
_, _ = ox.plot.plot_graph(G, node_size=0)

## 2. Select cycleways with a custom filter

Select ways tagged `highway=cycleway` with osmium. `BackReferenceWriter` includes their referenced nodes, and `remove_tags=False` preserves the nodes' tags.

Use `network_type=None` to load all selected ways and `retain_all=True` to keep disconnected components.

In [ ]:
# select cycleways and write them with their referenced nodes
filtered = Path("./data/dc-cycleways.osm.pbf")
tag_filter = osmium.filter.TagFilter(("highway", "cycleway"))
with osmium.BackReferenceWriter(
    filtered, ref_src=source, remove_tags=False, overwrite=True
) as writer:
    for way in osmium.FileProcessor(source, entities=osmium.osm.WAY).with_filter(tag_filter):
        writer.add_way(way)

In [ ]:
G = ox.graph.graph_from_pbf(filtered, network_type=None, retain_all=True)
_, _ = ox.plot.plot_graph(G, node_size=0)

## 3. Combine conditions in a custom bicycle filter

Select cycle streets, cycleways, living streets, designated bicycle paths, and roads with cycle tracks. These custom conditions select bicycle infrastructure independently of the built-in `bike` preset.

In [ ]:
def bicycle_filter(tags: dict[str, str]) -> bool:
    """
    Select ways with the desired bicycle infrastructure tags.

    Parameters
    ----------
    tags
        OSM tags for one way.

    Returns
    -------
    selected
        True if the way matches any of the selection conditions.
    """
    # include cycle streets, cycleways, and living streets
    highway = tags.get("highway", "")
    if "cyclestreet" in tags or "cycleway" in highway or "living_street" in highway:
        return True

    # include paths explicitly designated for bicycles
    if "path" in highway and "designated" in tags.get("bicycle", ""):
        return True

    # check cycle tracks on either side of the road
    keys = ("cycleway", "cycleway:right", "cycleway:left", "cycleway:both")
    return any("track" in tags.get(key, "") for key in keys)

In [ ]:
# apply the custom filter and include the selected ways' referenced nodes
filtered = Path("./data/dc-bicycle-network.osm.pbf")
with osmium.BackReferenceWriter(
    filtered, ref_src=source, remove_tags=False, overwrite=True
) as writer:
    for way in osmium.FileProcessor(source, entities=osmium.osm.WAY):
        if bicycle_filter(dict(way.tags)):
            writer.add_way(way)

In [ ]:
G = ox.graph.graph_from_pbf(filtered, network_type=None, retain_all=True)
_, _ = ox.plot.plot_graph(G, node_size=0)